# Gradient Boosting (Turkey, Pipeline B)

**Model**: `sklearn.ensemble.GradientBoostingRegressor` / **Target**: `ngdprsaxdctrq`  
**Variables**: Cat3 (22 vars + 3 COVID = 25 total)  
**Train**: 1995-Q2 to 2011-Q4 / **Val**: 2012-Q1 to 2017-Q4 / **Test**: 2018-Q1 to 2025-Q4 (32 quarters)  
**Scaling**: NO / **HP tuning**: YES — GridSearchCV(TimeSeriesSplit(5)) on train+val / **n_lags**: 6  
**Note**: Turkey keeps n_lags=6 (original). With 22 Cat3 vars: 22×7+3=157 features / 67 obs = 2.34 ratio.  
**10-model averaging** / **n_jobs=1** (Windows constraint)

In [1]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

sys.path.insert(0, os.path.join(os.path.pardir, 'turkey_data'))
from turkey_helpers import (
    gen_lagged_data, flatten_data, mean_fill_dataset,
    get_features, load_data,
    PREDICTIONS_DIR, TARGET, COVID
)

SEED = 42; np.random.seed(SEED)
TRAIN_START = '1995-01-01'; TRAIN_END = '2011-12-31'; VAL_END = '2017-12-31'
TEST_START  = '2018-01-01'; TEST_END  = '2025-12-31'
N_LAGS = 6; N_MODELS = 10  # Turkey keeps original n_lags=6 (22 vars, manageable ratio)
VINTAGES = {'m1': -2, 'm2': -1, 'm3': 0}

print('GB (Turkey) — Cat3, n_lags={}, scaling=NO, 10-model avg'.format(N_LAGS))

GB (Turkey) — Cat3, n_lags=6, scaling=NO, 10-model avg


In [2]:
monthly, _, metadata = load_data()
features = get_features('cat3', with_covid=True)
print('Features: {} + lags = {} lagged features per quarter'.format(len(features), len(features)*(N_LAGS+1)))

# Phase A: HP tuning on train+val (1995-2017)
tune_data   = monthly[(monthly['date'] >= TRAIN_START) & (monthly['date'] <= VAL_END)].reset_index(drop=True)
tune_filled = mean_fill_dataset(tune_data, tune_data)
tune_flat   = flatten_data(tune_filled, TARGET, N_LAGS)
tune_flat   = tune_flat.loc[tune_flat.date.dt.month.isin([3,6,9,12]), :].dropna(axis=0, how='any').reset_index(drop=True)

feature_cols = [
    c for c in tune_flat.columns
    if c != 'date' and c != TARGET
    and any(c == f or c.startswith(f + '_') for f in features)
]
print('Effective feature columns: {}, training quarters: {}'.format(len(feature_cols), len(tune_flat)))

tscv      = TimeSeriesSplit(n_splits=5)
gb_base   = GradientBoostingRegressor(random_state=SEED)
gb_grid   = {'n_estimators': [100, 300], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1], 'subsample': [0.8, 1.0]}
gb_search = GridSearchCV(gb_base, gb_grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=1)
gb_search.fit(tune_flat[feature_cols].values, tune_flat[TARGET].values)
best_params = gb_search.best_params_
print('Best GB params: {}'.format(best_params))

Features: 25 + lags = 175 lagged features per quarter


Effective feature columns: 175, training quarters: 90

Best GB params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}


In [3]:
# Phase B: Rolling test loop with 10-model averaging
test_quarters = monthly[
    (monthly['date'] >= TEST_START) &
    (monthly['date'] <= TEST_END) &
    monthly['date'].dt.month.isin([3,6,9,12])
]['date'].tolist()

predictions = {v: [] for v in VINTAGES}; actuals_list = []; failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actuals_list.append(monthly[monthly['date'] == pd_q][TARGET].values[0])

    for vn, offset in VINTAGES.items():
        fc = pd_q + pd.DateOffset(months=offset)
        train    = monthly[(monthly['date'] >= TRAIN_START) & (monthly['date'] <= fc)].reset_index(drop=True)
        train_m  = gen_lagged_data(metadata, train.copy(), fc, lag=0)
        train_f  = mean_fill_dataset(train_m, train_m)
        train_fl = flatten_data(train_f, TARGET, N_LAGS)
        train_fl = train_fl.loc[train_fl.date.dt.month.isin([3,6,9,12]), :].dropna(axis=0, how='any').reset_index(drop=True)

        if len(train_fl) < 10:
            predictions[vn].append(np.nan); continue

        try:
            models = []
            for k in range(N_MODELS):
                m = GradientBoostingRegressor(
                    n_estimators=best_params['n_estimators'],
                    max_depth=best_params['max_depth'],
                    learning_rate=best_params['learning_rate'],
                    subsample=best_params['subsample'],
                    random_state=SEED+k)
                m.fit(train_fl[feature_cols].values, train_fl[TARGET].values)
                models.append(m)

            tr     = monthly[monthly['date'] == fc].reset_index(drop=True)
            tr_m   = gen_lagged_data(metadata, tr.copy(), fc, lag=0)
            tr_f   = mean_fill_dataset(train_m, tr_m)
            ctx    = train_f.tail(N_LAGS + 1).iloc[:-1].copy()
            cmb    = pd.concat([ctx, tr_f], ignore_index=True)
            cmb_fl = flatten_data(cmb, TARGET, N_LAGS)
            tr_fl  = cmb_fl.tail(1)
            preds  = [m.predict(tr_fl[feature_cols].values)[0] for m in models]
            predictions[vn].append(np.nanmean(preds))
        except Exception:
            predictions[vn].append(np.nan); failed += 1

    if (i+1) % 8 == 0:
        print('  {} / {}'.format(i+1, len(test_quarters)))

print('Done. {} failures.'.format(failed))

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

...

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

  32 / 32
Done. 0 failures.


C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'nan' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  lagged_data.loc[(len(lagged_data) - pub_lag + lag - 1):, col] = np.nan
C:\Users\asus\Desktop\nowcasting_benchmark-main\nowcasting_benchmark-main\turkey_model_notebooks\..\turkey_data\..\data\helpers.py:131: FutureWarning: Setting an 

In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    df = pd.DataFrame({'date': test_quarters, 'actual': actuals_list, 'prediction': predictions[vn]})
    path = os.path.join(PREDICTIONS_DIR, 'gb_{}.csv'.format(vn))
    df.to_csv(path, index=False)
    print('Saved {} ({} rows) pred=[{:.4f},{:.4f}]'.format(
        os.path.basename(path), len(df), df['prediction'].min(), df['prediction'].max()))

Saved gb_m1.csv (32 rows) pred=[-0.0141,0.0263]
Saved gb_m2.csv (32 rows) pred=[-0.0223,0.0369]
Saved gb_m3.csv (32 rows) pred=[0.0085,0.0155]


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m]-p[m])**2)) if m.sum()>0 else np.nan

def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m]-p[m])) if m.sum()>0 else np.nan

panels = {
    'Pre-crisis (2018-2019)': ('2018-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2018-2025)': ('2018-01-01', '2025-12-31'),
}
a = np.array(actuals_list); d = pd.to_datetime(test_quarters)
print('GB (Turkey) — Evaluation by Panel & Vintage')
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print('--- {} ---'.format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print('  {}  RMSFE={:.6f}  MAE={:.6f}  N={}'.format(vn, rmse(a[m],pp[m]), mae(a[m],pp[m]), m.sum()))

GB (Turkey) — Evaluation by Panel & Vintage
--- Pre-crisis (2018-2019) ---
  m1  RMSFE=0.013765  MAE=0.010708  N=8
  m2  RMSFE=0.016294  MAE=0.013851  N=8
  m3  RMSFE=0.017243  MAE=0.013188  N=8
--- COVID      (2020-2021) ---
  m1  RMSFE=0.079726  MAE=0.054250  N=7
  m2  RMSFE=0.070223  MAE=0.043843  N=7
  m3  RMSFE=0.068511  MAE=0.042520  N=7
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.017138  MAE=0.012124  N=16
  m2  RMSFE=0.013646  MAE=0.010691  N=16
  m3  RMSFE=0.011688  MAE=0.008038  N=16
--- Full test  (2018-2025) ---
  m1  RMSFE=0.039880  MAE=0.021030  N=32
  m2  RMSFE=0.035223  MAE=0.018678  N=32
  m3  RMSFE=0.034237  MAE=0.016913  N=32
